# Agent Communication and Coordination Patterns

**Level:** Advanced · **Time:** 60 min

In this notebook, we move beyond the theory of multi-agent communication and implement the core architectural patterns (Sequential, Hierarchical, Swarm, and Blackboard) using industry-standard SDKs: **CrewAI**, **LangGraph**, and **AutoGen**. 

We will apply each pattern to the **Northstar Incident Response** scenario:
> *Northstar’s EU checkout conversion falls 31% shortly after a deployment.* 

> **Note:** The code blocks in this notebook are written to be fully syntactically correct against their respective SDKs. To allow this notebook to run without requiring 3 different API keys from 3 different providers, the outputs are simulated as they would appear in production.

---
## Pattern 1: Sequential (Pipeline) with CrewAI

The **Sequential Pattern** is the simplest and most predictable. The output of one agent is passed directly as the input to the next.

**Framework of Choice:** [CrewAI](https://www.crewai.com/) excels at defining strict, sequential role-playing processes.

In our scenario, we will define an `Observability Agent` that gathers metrics, followed strictly by an `Impact Agent` that estimates SLA drops.

In [ ]:
from crewai import Agent, Task, Crew, Process

# 1. Define Agents
observability_agent = Agent(
    role='Observability Specialist',
    goal='Gather metrics and logs regarding the EU Checkout drop.',
    backstory='Expert in Prometheus and Datadog.',
    verbose=True,
    allow_delegation=False
)

impact_agent = Agent(
    role='Impact Specialist',
    goal='Determine customer SLA impact based on observability metrics.',
    backstory='Expert in customer success and SLA contracts.',
    verbose=True,
    allow_delegation=False
)

# 2. Define Sequential Tasks
gather_metrics = Task(
    description='Find anomalies in EU checkout latency over the last 15 minutes.',
    expected_output='A summary of latency and error rates.',
    agent=observability_agent
)

assess_impact = Task(
    description='Using the metrics summary, calculate how many customers breached SLA.',
    expected_output='A final SLA impact report.',
    agent=impact_agent
)

# 3. Create Crew with Sequential Process
incident_crew = Crew(
    agents=[observability_agent, impact_agent],
    tasks=[gather_metrics, assess_impact],
    process=Process.sequential # Strictly A -> B
)

# execution_result = incident_crew.kickoff()

[2026-08-13 14:00:01][INFO]: Working Agent: Observability Specialist
[2026-08-13 14:00:03][INFO]: Task Output: Found 504 Gateway Timeouts in EU-West-1 correlating with deploy-842.
[2026-08-13 14:00:03][INFO]: Working Agent: Impact Specialist
[2026-08-13 14:00:06][INFO]: Task Output: 4,500 users affected. SLA breached by 2.1%.


**Selection Criteria:** Use Sequential patterns for linear workflows (e.g., ETL, content generation -> editing). Do *not* use it if the second agent needs to dynamically request more information from the first (as it's strictly one-way).

---
## Pattern 2: Hierarchical (Supervisor) with LangGraph

In the **Hierarchical Pattern**, a top-level Orchestrator or Supervisor manages a team of worker agents. It decomposes tasks and delegates them, maintaining the state of the overall objective.

**Framework of Choice:** [LangGraph](https://langchain-ai.github.io/langgraph/) is state-of-the-art for defining deterministic routing and stateful supervisor workflows as Directed Acyclic Graphs (DAGs).

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langgraph.graph import StateGraph, START, END

# 1. Define the Global State
class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    next_worker: str

# 2. Define Node Functions
def supervisor_node(state: AgentState):
    # Logic to decide who works next based on the messages
    if "metrics" not in str(state["messages"]):
        return {"next_worker": "observability"}
    elif "release" not in str(state["messages"]):
        return {"next_worker": "deployment"}
    return {"next_worker": "FINISH"}

def observability_node(state: AgentState):
    return {"messages": ["Observability: 504 errors found."]}

def deployment_node(state: AgentState):
    return {"messages": ["Deployment: deploy-842 rolled out 5 mins ago."]}

# 3. Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("Supervisor", supervisor_node)
workflow.add_node("observability", observability_node)
workflow.add_node("deployment", deployment_node)

# Add conditional edges from Supervisor to Workers
workflow.add_conditional_edges("Supervisor", lambda state: state["next_worker"], {
    "observability": "observability",
    "deployment": "deployment",
    "FINISH": END
})

# Workers always route back to the Supervisor
workflow.add_edge("observability", "Supervisor")
workflow.add_edge("deployment", "Supervisor")
workflow.set_entry_point("Supervisor")

app = workflow.compile()
# result = app.invoke({"messages": ["Incident: EU checkout drop"]})

Routing to: observability
Received: Observability: 504 errors found.
Routing to: deployment
Received: Deployment: deploy-842 rolled out 5 mins ago.
Routing to: FINISH
Final State: Incident: EU checkout drop -> Observability: 504 errors -> Deployment: deploy-842


**Selection Criteria:** Use Hierarchical when you need strict accountability, bounds, and deterministic routing. It prevents workers from going rogue, but the Supervisor can become a bottleneck.

---
## Pattern 3: Swarm (Decentralized) with AutoGen

In a **Swarm Pattern**, agents interact dynamically in a peer-to-peer fashion. They negotiate and hand off control freely based on the conversation context.

**Framework of Choice:** [AutoGen](https://microsoft.github.io/autogen/) excels at decentralized GroupChats where the LLM decides who speaks next.

In [ ]:
import autogen

# 1. Define Agents
observability_agent = autogen.AssistantAgent(
    name="Observability",
    system_message="You analyze metrics. Wait for alerts, then post findings."
)

deployment_agent = autogen.AssistantAgent(
    name="Deployment",
    system_message="You analyze releases. If you see metrics issues, check recent deploys."
)

critic_agent = autogen.AssistantAgent(
    name="Critic",
    system_message="You review the chat and declare mitigation ready when both agree."
)

user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=4
)

# 2. Create the GroupChat Swarm
groupchat = autogen.GroupChat(
    agents=[user_proxy, observability_agent, deployment_agent, critic_agent], 
    messages=[], 
    max_round=5
)
manager = autogen.GroupChatManager(groupchat=groupchat)

# execution
# user_proxy.initiate_chat(manager, message="EU Checkout dropped 31%. Investigate.")

User (to chat_manager): EU Checkout dropped 31%. Investigate.

Observability (to chat_manager): Looking at the dashboard. We have a massive spike in 504s starting exactly at 14:00 UTC.

Deployment (to chat_manager): 14:00 UTC matches exactly with the rollout of deploy-842 to the EU cluster. I recommend an immediate rollback.

Critic (to chat_manager): Both metrics and deployment timing align. Mitigation (Rollback) is supported and ready.


**Selection Criteria:** Use Swarms for highly creative or unstructured tasks (brainstorming, dynamic negotiation). Avoid for strict compliance workflows, as swarms can enter infinite loops or lose accountability without strict `max_round` turn limits.

---
## Pattern 4: Blackboard (Shared Memory Synthesis)

In the **Blackboard Pattern**, agents do not talk directly to each other (avoiding Swarm chaos and Supervisor bottlenecks). Instead, they asynchronously write scoped, provenance-tagged artifacts to a central datastore. A Critic agent independently evaluates the Blackboard.

This is the safest pattern for enterprise Incident Response, as it prevents agents from overwriting facts or treating another agent's hallucination as authority.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Artifact:
    role: str
    claim: str
    source: str
    confidence: float

@dataclass
class Blackboard:
    required_roles: set[str] = field(default_factory=lambda: {"observability", "deployment", "impact"})
    artifacts: dict[str, Artifact] = field(default_factory=dict)
    
    def publish(self, artifact: Artifact) -> None:
        if artifact.role not in self.required_roles:
            raise ValueError("Unauthorized role")
        self.artifacts[artifact.role] = artifact
        print(f"[Blackboard Write] {artifact.role} -> {artifact.source}")

    def evaluate(self) -> str:
        if set(self.artifacts.keys()) != self.required_roles:
            return "Incomplete: Missing evidence."
        
        # Critic synthesis logic
        if min(a.confidence for a in self.artifacts.values()) < 0.8:
            return "Escalate: Low confidence in evidence."
            
        return "Converged: Ready for Human Approval."

# Execution
board = Blackboard()

# Agents write asynchronously without knowing about each other
board.publish(Artifact("observability", "Spike in 504s", "datadog-link-1", 0.95))
board.publish(Artifact("deployment", "deploy-842 rolled out", "github-sha-abc", 0.99))

print("Critic Check 1:", board.evaluate())

board.publish(Artifact("impact", "4500 SLA breaches", "stripe-metrics", 0.85))

print("Critic Check 2:", board.evaluate())

[Blackboard Write] observability -> datadog-link-1
[Blackboard Write] deployment -> github-sha-abc
Critic Check 1: Incomplete: Missing evidence.
[Blackboard Write] impact -> stripe-metrics
Critic Check 2: Converged: Ready for Human Approval.


---
## Conclusion & Evaluation

When deciding which framework or pattern to use, always evaluate against a strong single-agent baseline.

- **Cost per Success:** Swarms often incur high token costs due to extensive context sharing.
- **Latency:** Blackboards allow full parallel execution (lowest P95 latency). Sequential has the highest latency.
- **Safety:** Hierarchical and Blackboard patterns enforce strong boundaries; Swarms are prone to context bleed.

**Golden Rule:** Never adopt a multi-agent framework simply to make a demo look cool. Add agents only when independent tools, disparate context windows, or separated accountability boundaries demonstrably improve your system's metrics.